In [2]:
import json
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
import requests
import os
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model= "llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=3000,
)

jsonLLM = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}}
)

In [ ]:
url = os.getenv("url")

def fetchVehicleData(vehicle_id, limit=10):
    if not vehicle_id:
        print("No vehicle_id provided.")
        return None

    api_key = os.getenv("DATA_API_KEY")
    
    params = {
        "vehicle_id": vehicle_id,
        "limit": limit,
        "sort": "desc"
    }
    
    headers = {
        "Authorization": f"Bearer {api_key}" if api_key else ""
    }
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        return response.json()
        
    except requests.exceptions.RequestException as e:
        print(f"API Request failed: {e}")
        return None

In [5]:
prompt = PromptTemplate(
    input_variables=["query"],
    template="""
Extract the vehicle ID (code/number) and vehicle name (model/brand) from the following query.

Return ONLY a valid JSON object with these exact keys:
- "vehicle_id": integer or null if not found
- "vehicle_name": string or null if not found

Query: {query}
"""
)

chain = prompt | jsonLLM
query = "Can you share a deep analysis for car id 34?"

response = chain.invoke({"query": query})

data = json.loads(response.content)

vehicle_id = data.get("vehicle_id")
vehicle_name = data.get("vehicle_name")

print(f"Vehicle : {vehicle_name}")
print(f"Vehicle id : {vehicle_id}")

Vehicle : None
Vehicle id : 34


In [6]:
rawVehicleData = fetchVehicleData(vehicle_id=vehicle_id)
print(f"Raw vehicle data: {rawVehicleData}")

Raw vehicle data: [[{'ID': '21892', 'repair_date': '2025-11-19', 'vendor_id': '1488', 'vehicle_id': '34', 'driver_id': '10503171', 'workorder_no': '12040', 'invoice_code': None, 'odometer': '359651', 'prev_id': None, 'diff_km': '4270', 'labor_subtotal': '0', 'parts_subtotal': '15876', 'total_amount': '15876', 'reference': None, 'description': None, 'account_id': None, 'archived': '1', 'branch_id': None, 'user_id': '151', 'created_at': '2025-11-20 10:29:50'}], [[{'item_id': '56018', 'parent_id': '21892', 'category': '1', 'task_id': '1', 'service_task': ' Engine Oil - Engine Oil - Engine oil ((Toyota Petron 10W30) (08Litr)', 'labor': '0', 'parts': '14826', 'subtotal': '14826'}], [{'item_id': '56019', 'parent_id': '21892', 'category': '1', 'task_id': '2', 'service_task': ' Oil Filter - Oil Filter', 'labor': '0', 'parts': '1050', 'subtotal': '1050'}]], [{'ID': '21893', 'repair_date': '2025-11-18', 'vendor_id': '1803', 'vehicle_id': '34', 'driver_id': '10503171', 'workorder_no': '12041', 'i

In [7]:
def parseVehicleData(rawVehicleData):
    """
    Parses alternating workorder headers and line item details into clean dicts.
    """
    if not rawVehicleData or not isinstance(rawVehicleData, list):
        return []

    parsedRecords = []
    
    for i in range(0, len(rawVehicleData) - 1, 2):
        header_list = rawVehicleData[i]
        items_list = rawVehicleData[i + 1]
        
        if not header_list:
            continue
            
        header = header_list[0]
        
        tasks = []
        for item_wrapper in items_list:
            if item_wrapper and isinstance(item_wrapper, list) and len(item_wrapper) > 0:
                item = item_wrapper[0]
                tasks.append({
                    "service_task": item.get("service_task", "").strip(),
                    "labor_cost": item.get("labor"),
                    "parts_cost": item.get("parts"),
                    "subtotal": item.get("subtotal")
                })
        
        parsedRecords.append({
            "workorder_no": header.get("workorder_no"),
            "repair_date": header.get("repair_date"),
            "odometer_km": header.get("odometer"),
            "distance_driven_since_last_service_km": header.get("diff_km"),
            "total_cost": header.get("total_amount"),
            "services_performed": tasks
        })

    return parsedRecords

In [8]:
vehicleData = parseVehicleData(rawVehicleData)

print(f"Parsed Vehicle Data : {vehicleData}")

Parsed Vehicle Data : [{'workorder_no': '12040', 'repair_date': '2025-11-19', 'odometer_km': '359651', 'distance_driven_since_last_service_km': '4270', 'total_cost': '15876', 'services_performed': [{'service_task': 'Engine Oil - Engine Oil - Engine oil ((Toyota Petron 10W30) (08Litr)', 'labor_cost': '0', 'parts_cost': '14826', 'subtotal': '14826'}, {'service_task': 'Oil Filter - Oil Filter', 'labor_cost': '0', 'parts_cost': '1050', 'subtotal': '1050'}]}, {'workorder_no': '12041', 'repair_date': '2025-11-18', 'odometer_km': '359712', 'distance_driven_since_last_service_km': '39364', 'total_cost': '13280', 'services_performed': [{'service_task': 'Others - Shafat Cross', 'labor_cost': '4080', 'parts_cost': '9200', 'subtotal': '13280'}]}, {'workorder_no': '12328', 'repair_date': '2025-12-20', 'odometer_km': '360920', 'distance_driven_since_last_service_km': '59436', 'total_cost': '168000', 'services_performed': [{'service_task': 'Tire Replacemnt - Dunliop (195R14C) (04 Tires)', 'labor_cost

In [13]:
vechicleDataPrompt = PromptTemplate(
    input_variables=["query", "vehicleData"],
    template="""
You are an expert automotive service advisor and maintenance analyst. Your goal is to analyze the provided vehicle maintenance logs, address the user's specific query, and provide a clear, professional summary alongside actionable recommendations.

### Instructions:
1. **Analyze Records:** Review the raw vehicle data and extract all key events, repair trends, and recurring mechanical issues.
2. **Address Query:** Directly answer the user's query using the maintenance context provided.
3. **Chronological Summary:** Present key services performed in chronological order, highlighting replaced parts and major inspections.
4. **Actionable Recommendations:** Identify overdue services, recurring issues, or potential preventive steps based on the vehicle's history.

---

### Input data:
User Query: {query}
Vehicle Data: {vehicleData}

---
### Output format:
Provide your response strictly in the following structure:

1. **Direct Answer / Overview**: Brief response tailored specifically to the user's query.
2. **Maintenance Summary**:
   - Organized chronologically (Date | Service Performed | Key Replaced Parts / Findings).
   - Highlight any noticeable patterns (e.g., frequent brake wear, repeated fluid leaks).
3. **Expert Recommendations & Next Steps**:
   - Brief, clear advice on upcoming maintenance or preventive actions needed.
   - Any warning signs or items to keep an eye on.
"""
)

parser = StrOutputParser()

vehicleChain = vechicleDataPrompt | llm | parser


vehicleSuggestion = vehicleChain.invoke(dict(
    query=query,
    vehicleData=json.dumps(vehicleData)
))
print(vehicleSuggestion)

### Direct Answer / Overview
The vehicle with ID 34 has a comprehensive maintenance history, with multiple services performed over the past year. The records indicate regular oil changes, tire replacements, and repairs to various components such as the clutch cylinder and brake system. To provide a detailed analysis, we will delve into the maintenance summary and expert recommendations.

### Maintenance Summary
Here is a chronological summary of the key services performed:
- 2025-11-18: Shafat Cross replacement (distance driven since last service: 39,364 km)
- 2025-11-19: Engine oil change (Toyota Petron 10W30, 8 liters) and oil filter replacement
- 2025-12-12: Clutch cylinder and brake oil replacement (distance driven since last service: 53,348 km)
- 2025-12-20: Tire replacement (Dunliop, 195R14C, 4 tires)
- 2026-01-07: Engine oil change (5 liters)
- 2026-03-11: Engine oil change (Delo Gold, 8 liters), oil filter replacement, and air filter replacement
- 2026-05-19: Engine oil change 